Created by Rishal on 05th July 2025. This program contains the recently designed algorithm for a validator to decide which Prepare messages to respond to with a success vote. 

In [14]:
import random
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

# The Algorithm

In [24]:
# Data Types
ParticipantID = int

@dataclass
class PrepareMsg:
    operatorID: int
    participants: List[int]
    isLocked: bool
    
    def __repr__(self):
        return f"PrepareMsg(op={self.operatorID}, parts={self.participants}, locked={self.isLocked})"

class Validator:
    def __init__(self):
        pass
    
    def operator(self, msg: PrepareMsg) -> int:
        """Extract operator ID from prepare message"""
        return msg.operatorID
    
    def isLocked(self, msg: PrepareMsg) -> bool:
        """Check if the interaction in message is locked by the operator"""
        return msg.isLocked
    
    def choose(self, prepare_msgs: List[PrepareMsg], view_random: int) -> Optional[PrepareMsg]:
        """
        Choose a single message from the list using view randomness ordering
        """
        if not prepare_msgs:
            return None
            
        # Step 1: Initialize arrays
        prepares_randomised = []
        prepares_locked_randomised = []
        
        # Step 2: Process each message
        for msg in prepare_msgs:
            # XOR operator ID with random view number
            operator_rand = self.operator(msg) ^ view_random
            prepares_randomised.append((operator_rand, msg))
            # If interaction is locked, add to locked list
            if self.isLocked(msg):
                prepares_locked_randomised.append((operator_rand, msg))
        
        # Print the randomized lists as requested
        print("preparesRandomised:", prepares_randomised)
        print("preparesLockedRandomised:", prepares_locked_randomised)
        
        # Step 3: Choose based on locked messages first
        if len(prepares_locked_randomised) > 0:
            # Sort locked messages and return highest
            prepares_locked_randomised.sort(key=lambda x: x[0])
            return prepares_locked_randomised[-1][1]  # Return the message part of highest tuple
        else:
            # Sort all messages and return highest
            prepares_randomised.sort(key=lambda x: x[0])
            return prepares_randomised[-1][1]  # Return the message part of highest tuple
    
    def prepare_timeout_handler(self, 
                              prepare_map: Dict[ParticipantID, List[PrepareMsg]], 
                              Q: List[ParticipantID],
                              view_random: int) -> List[PrepareMsg]:
        """
        Handle prepare timeout and determine which prepare messages to respond to
        
        Args:
            prepare_map: Dictionary mapping ParticipantID to list of PrepareMsg
            Q: List of participants in which contexts validator belongs
            view_random: The view's random number for ordering
            
        Returns:
            List of prepare messages to respond to
        """
        print(f"View Random Number: {view_random}")
        print(f"In Context of Participants: {Q}")
        for p in prepare_map.keys():
            print(f"Prepare Messages for Participant {p}: {prepare_map[p]}")
        print("\n")

        # Step 1: Initialize preparesToRespond
        prepares_to_respond = []
        
        # Step 2: For each participant in Q, choose one message
        for p in Q:
            if p in prepare_map:
                print(f"Choosing message for participant {p}")
                chosen_msg = self.choose(prepare_map[p], view_random)
                if chosen_msg:
                    prepares_to_respond.append(chosen_msg)
                    print(f"Chosen message for participant {p}: {chosen_msg}\n")
        
        # Printing messages to send Prepare vote and PrepareNil votes in response to
        print(f"Send Prepare Vote: {prepares_to_respond}")

        all_prepare_msgs, prepares_to_respond_fail = [], []
        for p in prepare_map:
            all_prepare_msgs.extend(prepare_map[p])
        for msg in all_prepare_msgs:
            if msg not in prepares_to_respond:
                prepares_to_respond_fail.append(msg)
        print(f"Send PrepareNil Vote: {prepares_to_respond_fail}")

        
    #     # Step 3: Process all prepare messages and send responses
    #     all_prepare_msgs = []
    #     for p in prepare_map:
    #         all_prepare_msgs.extend(prepare_map[p])
        
    #     print(f"\nProcessing responses for {len(all_prepare_msgs)} total messages:")
        
    #     for prepare_msg in all_prepare_msgs:
    #         if prepare_msg in prepares_to_respond:
    #             print(f"Responding to operator {self.operator(prepare_msg)} with PREPARE vote (containing lockedQCs)")
    #             self.send_prepare_vote(prepare_msg)
    #         else:
    #             print(f"Responding to operator {self.operator(prepare_msg)} with PREPARE_NIL vote")
    #             self.send_prepare_nil_vote(prepare_msg, prepare_map)
        
    #     return prepares_to_respond
    
    # def send_prepare_vote(self, prepare_msg: PrepareMsg):
    #     """Send Prepare vote containing lockedQCs"""
    #     # Implementation would send actual vote message
    #     print(f"  -> Sending PREPARE vote to operator {prepare_msg.operatorID}")
    
    # def send_prepare_nil_vote(self, prepare_msg: PrepareMsg, prepare_map: Dict[ParticipantID, List[PrepareMsg]]):
    #     """Send PrepareNil vote with proof of highest element"""
    #     # Implementation would send actual nil vote with proof
    #     print(f"  -> Sending PREPARE_NIL vote to operator {prepare_msg.operatorID}")

In [25]:
# Example usage and testing

# Create validator instance
validator = Validator()
    
# Example input
prepare_map = {
    1: [
        PrepareMsg(operatorID=10, participants=[1, 2], isLocked=False),
        PrepareMsg(operatorID=20, participants=[1, 3], isLocked=True),
        PrepareMsg(operatorID=30, participants=[1, 4], isLocked=False)
    ],
    2: [
        PrepareMsg(operatorID=40, participants=[2, 3], isLocked=True),
        PrepareMsg(operatorID=50, participants=[2, 4], isLocked=False)
    ],
    3: [
        PrepareMsg(operatorID=60, participants=[3, 4], isLocked=False)
    ]
}
    
Q = [1, 2, 3]  # Participants in whose contexts the validator belongs to
view_random = 66  # Example view random number
    
print("=== BFT Consensus Prepare Timeout Algorithm ===")
    
# Execute the prepare timeout handler
validator.prepare_timeout_handler(prepare_map, Q, view_random)  

=== BFT Consensus Prepare Timeout Algorithm ===
View Random Number: 66
In Context of Participants: [1, 2, 3]
Prepare Messages for Participant 1: [PrepareMsg(op=10, parts=[1, 2], locked=False), PrepareMsg(op=20, parts=[1, 3], locked=True), PrepareMsg(op=30, parts=[1, 4], locked=False)]
Prepare Messages for Participant 2: [PrepareMsg(op=40, parts=[2, 3], locked=True), PrepareMsg(op=50, parts=[2, 4], locked=False)]
Prepare Messages for Participant 3: [PrepareMsg(op=60, parts=[3, 4], locked=False)]


Choosing message for participant 1
preparesRandomised: [(72, PrepareMsg(op=10, parts=[1, 2], locked=False)), (86, PrepareMsg(op=20, parts=[1, 3], locked=True)), (92, PrepareMsg(op=30, parts=[1, 4], locked=False))]
preparesLockedRandomised: [(86, PrepareMsg(op=20, parts=[1, 3], locked=True))]
Chosen message for participant 1: PrepareMsg(op=20, parts=[1, 3], locked=True)

Choosing message for participant 2
preparesRandomised: [(106, PrepareMsg(op=40, parts=[2, 3], locked=True)), (112, PrepareMsg

# Testing

In [ ]:
# Example usage and testing
def run_test_case(test_name: str, validator: BFTConsensusValidator, prepare_map: Dict[ParticipantID, List[PrepareMsg]], Q: List[ParticipantID], view_random: int):
    """Run a single test case and print results"""
    print(f"\n{'='*50}")
    print(f"TEST CASE: {test_name}")
    print(f"{'='*50}")
    print(f"Q (participants): {Q}")
    print(f"view_random: {view_random}")
    print(f"prepare_map: {prepare_map}")
    print()
    
    result = validator.prepare_timeout_handler(prepare_map, Q, view_random)
    print(f"\nFINAL RESULT - Messages to respond to: {result}")
    return result

def main():
    """Comprehensive test suite for the BFT consensus prepare timeout algorithm"""
    
    validator = BFTConsensusValidator()
    
    # TEST CASE 1: Basic scenario with mixed locked/unlocked messages
    print("=== BFT Consensus Prepare Timeout Algorithm - Test Suite ===")
    
    test1_prepare_map = {
        1: [
            PrepareMsg(operatorID=10, participants=[1, 2], isLocked=False),
            PrepareMsg(operatorID=20, participants=[1, 3], isLocked=True),
            PrepareMsg(operatorID=30, participants=[1, 4], isLocked=False)
        ],
        2: [
            PrepareMsg(operatorID=40, participants=[2, 3], isLocked=True),
            PrepareMsg(operatorID=50, participants=[2, 4], isLocked=False)
        ],
        3: [
            PrepareMsg(operatorID=60, participants=[3, 4], isLocked=False)
        ]
    }
    run_test_case("Basic Mixed Scenario", validator, test1_prepare_map, [1, 2, 3], 66)
    
    # TEST CASE 2: All messages are locked
    test2_prepare_map = {
        1: [
            PrepareMsg(operatorID=100, participants=[1, 2], isLocked=True),
            PrepareMsg(operatorID=200, participants=[1, 3], isLocked=True)
        ],
        2: [
            PrepareMsg(operatorID=300, participants=[2, 3], isLocked=True),
            PrepareMsg(operatorID=400, participants=[2, 4], isLocked=True)
        ]
    }
    run_test_case("All Messages Locked", validator, test2_prepare_map, [1, 2], 123)
    
    # TEST CASE 3: No messages are locked
    test3_prepare_map = {
        1: [
            PrepareMsg(operatorID=15, participants=[1, 2], isLocked=False),
            PrepareMsg(operatorID=25, participants=[1, 3], isLocked=False),
            PrepareMsg(operatorID=35, participants=[1, 4], isLocked=False)
        ],
        2: [
            PrepareMsg(operatorID=45, participants=[2, 3], isLocked=False)
        ]
    }
    run_test_case("No Messages Locked", validator, test3_prepare_map, [1, 2], 999)
    
    # TEST CASE 4: Single participant, single message
    test4_prepare_map = {
        1: [
            PrepareMsg(operatorID=77, participants=[1], isLocked=True)
        ]
    }
    run_test_case("Single Participant Single Message", validator, test4_prepare_map, [1], 42)
    
    # TEST CASE 5: Empty prepare map
    test5_prepare_map = {}
    run_test_case("Empty Prepare Map", validator, test5_prepare_map, [1, 2], 888)
    
    # TEST CASE 6: Participant not in prepare map
    test6_prepare_map = {
        1: [
            PrepareMsg(operatorID=111, participants=[1, 2], isLocked=False)
        ]
    }
    run_test_case("Participant Not in Prepare Map", validator, test6_prepare_map, [1, 2, 3], 555)
    
    # TEST CASE 7: Large operator IDs to test XOR behavior
    test7_prepare_map = {
        1: [
            PrepareMsg(operatorID=0xFFFF, participants=[1, 2], isLocked=False),
            PrepareMsg(operatorID=0x0001, participants=[1, 3], isLocked=True),
            PrepareMsg(operatorID=0x8000, participants=[1, 4], isLocked=False)
        ]
    }
    run_test_case("Large Operator IDs", validator, test7_prepare_map, [1], 0xAAAA)
    
    # TEST CASE 8: Multiple participants with varying message counts
    test8_prepare_map = {
        1: [
            PrepareMsg(operatorID=5, participants=[1, 2, 3], isLocked=True)
        ],
        2: [
            PrepareMsg(operatorID=10, participants=[2, 3], isLocked=False),
            PrepareMsg(operatorID=20, participants=[2, 4], isLocked=False),
            PrepareMsg(operatorID=30, participants=[2, 5], isLocked=True)
        ],
        3: [
            PrepareMsg(operatorID=40, participants=[3, 4, 5], isLocked=False),
            PrepareMsg(operatorID=50, participants=[3, 6], isLocked=False)
        ],
        4: [
            PrepareMsg(operatorID=60, participants=[4, 5, 6, 7], isLocked=True),
            PrepareMsg(operatorID=70, participants=[4, 8], isLocked=True),
            PrepareMsg(operatorID=80, participants=[4, 9], isLocked=False),
            PrepareMsg(operatorID=90, participants=[4, 10], isLocked=True)
        ]
    }
    run_test_case("Varying Message Counts", validator, test8_prepare_map, [1, 2, 3, 4], 2024)
    
    # TEST CASE 9: Edge case with zero view random
    test9_prepare_map = {
        1: [
            PrepareMsg(operatorID=100, participants=[1], isLocked=False),
            PrepareMsg(operatorID=200, participants=[1], isLocked=True)
        ]
    }
    run_test_case("Zero View Random", validator, test9_prepare_map, [1], 0)
    
    # TEST CASE 10: Only locked messages vs only unlocked messages
    test10_prepare_map = {
        1: [
            PrepareMsg(operatorID=1, participants=[1], isLocked=True),
            PrepareMsg(operatorID=2, participants=[1], isLocked=True),
            PrepareMsg(operatorID=3, participants=[1], isLocked=True)
        ],
        2: [
            PrepareMsg(operatorID=4, participants=[2], isLocked=False),
            PrepareMsg(operatorID=5, participants=[2], isLocked=False),
            PrepareMsg(operatorID=6, participants=[2], isLocked=False)
        ]
    }
    run_test_case("Locked vs Unlocked Groups", validator, test10_prepare_map, [1, 2], 777)
    
    # TEST CASE 11: Same operator IDs but different locked status
    test11_prepare_map = {
        1: [
            PrepareMsg(operatorID=50, participants=[1, 2], isLocked=False),
            PrepareMsg(operatorID=50, participants=[1, 3], isLocked=True)  # Same operatorID
        ]
    }
    run_test_case("Same Operator IDs Different Lock Status", validator, test11_prepare_map, [1], 333)
    
    print(f"\n{'='*50}")
    print("TEST SUITE COMPLETED")
    print(f"{'='*50}")

if __name__ == "__main__":
    main()